---
# `LangChain Important Components`
---

1. Models
2. Prompts
3. Chains
4. Memory
5. Indexes
6. Agents

### Models
- LLMs - Large Language Model
- Embeddings Models
- LIM - Large Image Model
- MultiModal 

# Components of LangChain

LangChain is easier to understand if you think of it as an **orchestration framework around an LLM**.

An LLM can generate text, but a real AI application usually needs much more:

```text
User
 │
 ▼
Prompt
 │
 ▼
LLM
 │
 ├── Documents
 ├── Embeddings
 ├── Vector Database
 ├── Retriever
 ├── Tools
 ├── Agent
 ├── Memory / Conversation History
 └── Output Parser
 │
 ▼
Final Response
```

The major LangChain components are:

1. **Models**
2. **Prompt Templates**
3. **Messages**
4. **Output Parsers / Structured Output**
5. **Document Loaders**
6. **Text Splitters**
7. **Embeddings**
8. **Vector Stores**
9. **Retrievers**
10. **Chains / Runnables**
11. **Tools**
12. **Agents**
13. **Conversation History**
14. **Callbacks / Observability**

---

# 1. Models

The **model** is the actual LLM or embedding model that performs computation.

LangChain can work with models from different providers.

Examples:

* OpenAI
* Anthropic
* Google Gemini
* Mistral
* Hugging Face
* Local models

There are two important categories.

### Chat Models

Used for conversations and instruction following.

```python
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1-mini"
)

response = model.invoke(
    "Explain Machine Learning"
)

print(response.content)
```

The important idea is:

```text
Application
     ↓
LangChain
     ↓
Chat Model
     ↓
LLM Provider
```

---

# 2. Prompt Templates

A **Prompt Template** creates reusable prompts.

Instead of hardcoding:

```text
Explain Python for a beginner.
```

we create:

```text
Explain {topic} for a {level}.
```

Then:

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} for a {level}."
)

messages = prompt.invoke({
    "topic": "Python",
    "level": "beginner"
})
```

This produces a structured prompt that can be passed to the model.

### Why useful?

Prompt templates make prompts:

* Reusable
* Consistent
* Dynamic
* Easier to maintain

---

# 3. Messages

Chat models work with different types of messages.

The most common are:

### System Message

Defines behavior.

```text
You are a helpful Python teacher.
```

### Human Message

The user's request.

```text
Explain decorators.
```

### AI Message

A previous assistant response.

```text
Decorators are functions that...
```

Conceptually:

```text
System
   ↓
Human
   ↓
AI
   ↓
Human
   ↓
AI
```

This structure is important for conversational applications.

---

# 4. Output Parsers / Structured Output

LLMs normally return text.

For example:

```text
Name: Arun
Age: 25
Role: AI Engineer
```

But applications often need structured data:

```json
{
  "name": "Arun",
  "age": 25,
  "role": "AI Engineer"
}
```

LangChain provides mechanisms for structured output and output parsing.

For example, with a Pydantic schema:

```python
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

class Person(BaseModel):
    name: str
    age: int
    role: str

model = ChatOpenAI(
    model="gpt-4.1-mini"
).with_structured_output(Person)

result = model.invoke(
    "My name is Arun, I am 25 and I work as an AI Engineer."
)

print(result)
```

Now your application gets structured data rather than having to manually parse arbitrary text.

---

# 5. Document Loaders

Document Loaders bring external data into your LangChain application.

Examples:

```text
PDF
DOCX
TXT
CSV
HTML
Web pages
Cloud storage
```

For example:

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("machine_learning.pdf")

documents = loader.load()
```

The result is a collection of `Document` objects containing content and metadata.

---

# 6. Text Splitters

Suppose your PDF contains:

```text
500 pages
```

You generally don't want to send the entire document to the model in one request.

Instead:

```text
500 Pages
    ↓
Text Extraction
    ↓
Text Splitting
    ↓
Chunk 1
Chunk 2
Chunk 3
...
Chunk N
```

Example:

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
```

### Why chunking?

Because it helps with:

* Context-window limits
* Retrieval quality
* Embedding efficiency
* Cost
* Latency

---

# 7. Embeddings

An **embedding model** converts text into numerical vectors.

```text
"Machine Learning"
       ↓
Embedding Model
       ↓
[0.21, -0.54, 0.83, ...]
```

The vector represents semantic information.

Example:

```text
"Machine Learning"
        ↓
[0.2, 0.7, -0.1, ...]

"Deep Learning"
        ↓
[0.3, 0.6, -0.2, ...]
```

Their vectors may be close because their meanings are related.

---

# 8. Vector Stores

A **Vector Store** stores embeddings and allows similarity search.

Common choices include:

* FAISS
* Chroma
* Pinecone
* Qdrant
* Weaviate
* Milvus

The workflow:

```text
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Vector Store
```

Example:

```python
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)
```

---

# 9. Retrievers

A **Retriever** finds the documents relevant to a user's question.

Suppose your PDF contains information about:

```text
Machine Learning
Deep Learning
Transformers
CNN
RNN
```

User asks:

```text
What is backpropagation?
```

The retriever searches the vector store and returns the most relevant chunks.

```text
Question
   ↓
Retriever
   ↓
Relevant Chunks
   ↓
LLM
```

Example:

```python
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke(
    "What is backpropagation?"
)
```

---

# 10. Chains / Runnables

A **chain** connects multiple operations into a pipeline.

For example:

```text
Question
   ↓
Prompt
   ↓
LLM
   ↓
Output Parser
```

Modern LangChain uses the **Runnable** abstraction heavily for composing these steps.

Example:

```python
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

result = chain.invoke({
    "topic": "Transformers",
    "level": "beginner"
})
```

The `|` operator means:

```text
Prompt
   ↓
Model
   ↓
Parser
```

---

# 11. Tools

A **Tool** gives an LLM access to an external function or system.

Examples:

```text
Calculator
Search Engine
Python
SQL Database
Weather API
Calendar
GitHub
Custom API
```

For example:

```text
User
 │
 ▼
LLM
 │
 ▼
Calculator Tool
 │
 ▼
Result
 │
 ▼
LLM
 │
 ▼
Answer
```

Without a calculator:

```text
LLM → predicts an answer
```

With a calculator:

```text
LLM → calls calculator → receives exact result → responds
```

---

# 12. Agents

An **Agent** uses an LLM to decide **which tool or action to use**.

This is different from a fixed chain.

### Chain

You define:

```text
Step 1
 ↓
Step 2
 ↓
Step 3
```

### Agent

The model decides:

```text
User Request
      ↓
    Agent
      │
 ┌────┼─────┐
 ▼    ▼     ▼
Search SQL Calculator
```

For example:

```text
User:
What is 25 × 42 and what is today's weather?
```

The agent might decide:

```text
25 × 42
   ↓
Calculator

Weather
   ↓
Weather API
```

Then combine the results into one response.

---

# 13. Conversation History

LLMs are generally stateless between separate model calls unless the application sends previous messages again or uses some external state/memory mechanism.

For example:

```text
User:
My name is Arun.

Assistant:
Nice to meet you.

User:
What's my name?
```

The application needs to provide the previous conversation context:

```text
System
Human: My name is Arun.
AI: Nice to meet you.
Human: What's my name?
```

The model can then answer:

```text
Arun.
```

For modern LangChain applications, **conversation history is typically managed explicitly**, rather than treating "memory" as a magical permanent capability.

---

# 14. Callbacks and Observability

Complex LLM applications need monitoring.

You may want to know:

```text
Which model was called?
How long did it take?
How many tokens were used?
Which retriever returned the documents?
Which tool was called?
Where did an error occur?
```

LangChain provides callback and tracing integrations that help developers inspect and debug these workflows.

For production systems, observability is extremely important.

---

# Putting Everything Together

Consider a **PDF Chatbot**.

The user asks:

```text
What is the company's refund policy?
```

The application can work like this:

```text
                    PDF
                     │
                     ▼
              Document Loader
                     │
                     ▼
                Text Splitter
                     │
                     ▼
                 Embeddings
                     │
                     ▼
                Vector Store
                     │
                     │
User Question ───────┤
                     ▼
                 Retriever
                     │
                     ▼
              Relevant Chunks
                     │
                     ▼
              Prompt Template
                     │
                     ▼
                  LLM
                     │
                     ▼
              Output Parser
                     │
                     ▼
                Final Answer
```

This is the foundation of a **RAG application**.

---

# Another Example: AI Agent

Suppose the user says:

> "Find my order and tell me when it will arrive."

Architecture:

```text
User
 │
 ▼
Agent
 │
 ├──── Order Database
 │
 ├──── Shipping API
 │
 └──── Calculator
 │
 ▼
LLM
 │
 ▼
Final Answer
```

Here:

* **LLM** → reasoning/decision-making
* **Agent** → chooses actions
* **Tools** → perform actions
* **Database/API** → provide external information

---

# How the Components Relate

The easiest way to remember them is:

```text
                 LANGCHAIN
                     │
        ┌────────────┼────────────┐
        │            │            │
        ▼            ▼            ▼
      Models       Prompts       Tools
        │            │            │
        └────────────┼────────────┘
                     │
                     ▼
                 Runnables
                     │
        ┌────────────┼─────────────┐
        ▼            ▼             ▼
   Documents     Embeddings     Agents
        │            │             │
        ▼            ▼             ▼
     Chunks      Vector Store    Actions
                     │
                     ▼
                 Retriever
                     │
                     ▼
                    LLM
                     │
                     ▼
                  Response
```

---

# Components You Should Learn First

Since you're learning **GenAI engineering**, don't try to learn every LangChain component at once.

Use this order:

### Level 1 — Foundation

```text
1. Chat Models
2. Prompt Templates
3. Messages
4. Structured Output
5. Runnables
```

### Level 2 — RAG

```text
6. Document Loaders
7. Text Splitters
8. Embeddings
9. Vector Stores
10. Retrievers
```

### Level 3 — Agents

```text
11. Tools
12. Tool Calling
13. Agents
14. Agent State / Workflows
```

### Level 4 — Production

```text
15. Streaming
16. Callbacks
17. Tracing
18. Evaluation
19. Error Handling
20. Deployment
```

---

# The Most Important Concept

Don't think of LangChain as an LLM.

Think of it as:

```text
LLM
+
Data
+
Retrieval
+
Tools
+
Workflow
+
Application Logic
```

LangChain **orchestrates these components** to build an AI application.

---

# LangChain vs LLM

| LLM                                    | LangChain                                 |
| -------------------------------------- | ----------------------------------------- |
| Generates/predicts language            | Orchestrates AI workflows                 |
| Learns from training data              | Connects models to external systems       |
| GPT, Claude, Gemini, etc.              | Framework/library                         |
| Provides language intelligence         | Provides application infrastructure       |
| Cannot inherently access your database | Can connect your application to databases |
| Model                                  | Application framework                     |

---

# Final Mental Model

Remember this architecture:

```text
                 USER
                   │
                   ▼
             APPLICATION
                   │
                   ▼
               LANGCHAIN
                   │
       ┌───────────┼───────────┐
       │           │           │
       ▼           ▼           ▼
     PROMPT       LLM        TOOLS
                   │           │
                   │           ├── API
                   │           ├── SQL
                   │           └── Python
                   │
                   ▼
               RETRIEVER
                   │
                   ▼
              VECTOR STORE
                   │
                   ▼
               DOCUMENTS
                   │
                   ▼
              FINAL RESPONSE
```

**In one sentence:** LangChain provides the building blocks and orchestration layer that connects **LLMs + prompts + data + retrieval + tools + agents + application logic** into complete GenAI applications.


## `Detailed Overview of Componets` 

# 1. Clean Jupyter Notebook Notes

# Overview of LangChain Components and Models

## What is LangChain?

**LangChain** is a framework for building applications powered by Large Language Models (LLMs).

It provides reusable components that help us connect:

```text
User
  ↓
Prompt
  ↓
Model
  ↓
Tools / Retrieval / Memory
  ↓
Application
```

LangChain supports integrations with many model providers and provides higher-level abstractions for building LLM applications and agents. ([Docs by LangChain][1])

---

# LangChain Components

A typical LangChain application can be understood through these major components:

```text
                    LangChain
                       │
       ┌───────────────┼────────────────┐
       ↓               ↓                ↓
    Models          Prompts           Tools
       │               │                │
       └───────────────┼────────────────┘
                       ↓
                    Chains
                       ↓
                   Retrieval
                       ↓
                    Agents
                       ↓
                    Memory
```

Let's understand them one by one.

---

## 1. Models

### Definition

A **model** is the AI system that performs tasks such as:

* Understanding text
* Generating text
* Answering questions
* Summarizing
* Reasoning
* Generating structured output
* Calling tools
* Understanding images/audio, depending on the model

In LangChain, models provide a standardized interface so applications can work with different model providers. ([Docs by LangChain][1])

Examples of model providers include:

* OpenAI
* Anthropic
* Google
* Mistral
* Ollama
* Other supported providers

### Simple Example

```text
User Question
     ↓
   Model
     ↓
AI Response
```

---

# 2. Prompts

## Definition

A **prompt** is the instruction or input given to a model.

For example:

```text
You are an expert Python teacher.

Explain the following concept in simple language:

{topic}
```

Instead of manually creating strings, LangChain provides prompt templates that allow dynamic values.

### Example

```text
Template:
"Explain {topic} in simple language."

Input:
topic = "RAG"

Output Prompt:
"Explain RAG in simple language."
```

### Why Prompts Matter

Good prompts can help control:

* Model behavior
* Response format
* Context
* Instructions
* Output style

---

# 3. Messages

Modern chat models work with different types of messages.

Common message roles include:

### System Message

Defines the behavior or instructions for the model.

```text
You are a helpful AI tutor.
```

### Human Message

Represents the user's input.

```text
Explain RAG.
```

### AI Message

Represents the model's response.

```text
RAG stands for Retrieval-Augmented Generation...
```

### Tool Message

Contains the result returned by a tool.

```text
Weather API:
Delhi = 32°C
```

---

# 4. Tools

## Definition

A **tool** is a function that an AI application can call to perform an action or retrieve information.

Examples:

```text
Calculator
Weather API
Database
Web Search
Python Function
Email API
CRM API
```

### Example

A user asks:

```text
What is the weather in Delhi?
```

The model may decide:

```text
User Question
     ↓
    LLM
     ↓
Weather Tool
     ↓
Weather API
     ↓
Result
     ↓
    LLM
     ↓
Final Answer
```

This is important because an LLM by itself does not automatically have access to every external system.

---

# 5. Retrievers

## Definition

A **retriever** searches a knowledge source and returns relevant information for a user's query.

It is commonly used in **RAG (Retrieval-Augmented Generation)**.

### Example

Suppose we have:

```text
1000 company documents
```

User asks:

```text
What is our company's leave policy?
```

Instead of sending all 1000 documents to the model:

```text
User Query
    ↓
Retriever
    ↓
Relevant Documents
    ↓
LLM
    ↓
Answer
```

The retriever finds the most relevant documents.

---

# 6. Document Loaders

## Definition

**Document loaders** load data from different sources into a format that can be processed by a LangChain application.

Possible sources include:

* PDF
* CSV
* Web pages
* Text files
* Databases
* Cloud storage
* Other data sources

Example:

```text
PDF
 ↓
Document Loader
 ↓
Documents
 ↓
Text Processing
 ↓
Embeddings
 ↓
Vector Store
```

---

# 7. Text Splitters

Large documents are usually divided into smaller pieces called **chunks**.

Example:

```text
Large PDF
   ↓
Text Splitter
   ↓
Chunk 1
Chunk 2
Chunk 3
Chunk 4
```

### Why?

LLMs and retrieval systems work better when relevant pieces of information can be retrieved independently.

Text splitting is especially important in RAG pipelines.

---

# 8. Embeddings

## Definition

**Embeddings convert text or other supported data into numerical vectors that represent semantic meaning.**

Example:

```text
"Python is a programming language"
                 ↓
          Embedding Model
                 ↓
[0.12, -0.42, 0.87, ...]
```

Semantically similar text tends to have vectors that are close together in embedding space.

### Used For

* Semantic search
* RAG
* Document similarity
* Recommendation systems
* Clustering
* Retrieval

---

# 9. Vector Stores

## Definition

A **vector store** stores embeddings and allows us to search for vectors that are semantically similar to a query.

Example:

```text
Documents
    ↓
Embeddings
    ↓
Vector Store
    ↓
Similarity Search
    ↓
Relevant Documents
```

Examples of vector databases/stores include:

* Chroma
* FAISS
* Pinecone
* Weaviate
* Milvus

---

# 10. Chains

## Definition

A **chain** connects multiple operations into a predefined sequence.

For example:

```text
User Input
    ↓
Prompt
    ↓
LLM
    ↓
Parser
    ↓
Final Output
```

The key idea is:

> **Chain = predefined sequence of operations.**

Chains are useful when the workflow is relatively predictable.

---

# 11. Agents

## Definition

An **agent** is an application where the model can decide what actions or tools to use to accomplish a task.

Example:

```text
User:
"What is the weather in Delhi and convert
the temperature to Fahrenheit?"

             ↓
           Agent
             ↓
      ┌──────┴──────┐
      ↓             ↓
Weather Tool    Calculator
      ↓             ↓
   32°C          89.6°F
      └──────┬──────┘
             ↓
        Final Answer
```

### Chain vs Agent

**Chain:**

```text
A → B → C → D
```

The path is mostly predetermined.

**Agent:**

```text
       ┌→ Tool A
LLM ───┼→ Tool B
       └→ Tool C
```

The model can decide which tool/path to use.

---

# 12. Output Parsers / Structured Output

## Definition

Output handling converts model responses into a predictable format.

Instead of:

```text
The candidate has 5 years of experience...
```

we may want:

```json
{
  "name": "Arun",
  "experience": 5,
  "skills": ["Python", "LangChain"]
}
```

Structured output is particularly useful when an LLM's response needs to be consumed by application code.

---

# 13. Memory / State

LLM calls are generally stateless unless the application provides previous conversation information.

For example:

```text
User:
My name is Arun.

Assistant:
Nice to meet you, Arun.

User:
What is my name?
```

The application needs to provide the relevant conversation/state so the model can answer correctly.

Memory/state mechanisms are therefore important for conversational applications and agents.

---

# Models in LangChain

Now let's focus on the most important component: **Models**.

## What is a Model?

A model is the AI engine that receives input and produces an output.

```text
Input
  ↓
Model
  ↓
Output
```

For example:

```text
Input:
"Explain machine learning."

       ↓

     LLM

       ↓

Output:
"Machine learning is a..."
```

LangChain provides standardized interfaces for interacting with models from different providers. ([Docs by LangChain][1])

---

# Types of Models

In LangChain, the important model categories to understand are:

1. **Chat Models**
2. **Embedding Models**
3. **Traditional / Text Completion Models** — mainly relevant when working with older APIs or specific integrations.

---

# 1. Chat Models

## Definition

A **chat model** is designed to work with conversational messages.

Instead of simply receiving a string:

```text
"Explain RAG"
```

the model can receive structured messages:

```text
System → You are an AI tutor.

Human → Explain RAG.

AI → RAG stands for...
```

Chat models are the primary model abstraction used in modern LangChain applications.

---

## Chat Model Input

Conceptually:

```text
Messages
   ↓
Chat Model
   ↓
AI Message
```

### Example

```text
System:
You are a helpful AI teacher.

Human:
Explain embeddings.

        ↓

    Chat Model

        ↓

AI:
Embeddings are numerical
representations of meaning...
```

---

# Important Chat Model Capabilities

Modern chat models can support capabilities such as:

### 1. Text Generation

Generate natural-language responses.

### 2. Tool Calling

The model can request that an application execute a tool.

```text
LLM
 ↓
Tool Call
 ↓
Tool Result
 ↓
LLM
```

### 3. Structured Output

Return information in a predictable schema.

```text
LLM
 ↓
Structured JSON
```

### 4. Multimodality

Depending on the specific model, the model may process:

* Text
* Images
* Audio
* Other modalities

The exact capabilities depend on the model/provider. ([Docs by LangChain][1])

---

# Model Parameters

Different models expose parameters that control how they behave.

## 1. Temperature

**Temperature controls the randomness/variability of generated responses.**

Conceptually:

```text
Low temperature
     ↓
More predictable

High temperature
     ↓
More varied
```

For example:

### Low Temperature

Useful for:

* Classification
* Extraction
* Deterministic tasks
* Structured responses

### Higher Temperature

Can be useful for:

* Creative writing
* Brainstorming
* Generating diverse ideas

**Interview point:** Temperature does not make a model more intelligent. It primarily changes the sampling behavior of generation.

---

# 2. Max Output Tokens

Controls the maximum amount of output the model can generate.

```text
Max output tokens = 100
        ↓
Shorter possible response

Max output tokens = 2000
        ↓
Longer possible response
```

The exact token limits depend on the model.

---

# 3. Model Name

A model is identified by a provider-specific model identifier.

Conceptually:

```text
Provider : Model
```

For example, current LangChain documentation uses formats such as:

```text
openai:gpt-5.5
google_genai:gemini-3.5-flash
anthropic:claude-opus-4-6
```

The available model names and capabilities change over time, so the provider's current documentation should be checked when selecting a model. ([Docs by LangChain][1])

---

# 4. Streaming

Instead of waiting for the complete answer:

```text
................. Complete Answer
```

streaming returns pieces progressively:

```text
Hello
Hello Arun
Hello Arun, welcome
Hello Arun, welcome to...
```

This is useful for chat interfaces because users can see the response while it is being generated.

---

# 5. Tool Calling

A model can determine that an external tool is required.

Example:

```text
User:
What is 25 × 40?

        ↓

      Model
        ↓
Calculator Tool
        ↓
1000
        ↓
      Model
        ↓
Final Answer
```

The model itself does not necessarily perform the external action; it generates a structured tool call that the application executes.

---

# Embedding Models

Chat models and embedding models have **different purposes**.

## Chat Model

Used for:

```text
Question → Answer
```

## Embedding Model

Used for:

```text
Text → Vector
```

Example:

```text
"LangChain is an LLM framework"
              ↓
       Embedding Model
              ↓
[0.12, 0.43, -0.21, ...]
```

These vectors can then be stored in a vector store and used for semantic retrieval.

---

# Chat Model vs Embedding Model

| Feature      | Chat Model                        | Embedding Model               |
| ------------ | --------------------------------- | ----------------------------- |
| Main purpose | Generate/understand responses     | Convert data into vectors     |
| Input        | Messages/text, depending on model | Text or other supported input |
| Output       | AI response                       | Vector                        |
| Used for     | Chat, reasoning, agents           | Search, RAG, similarity       |
| Example      | GPT/Gemini/Claude                 | Text embedding model          |

### Easy Memory Trick

> **Chat Model = Think & Generate**
> **Embedding Model = Represent Meaning as Numbers**

---

# Example: Complete RAG Application

Let's connect the components.

```text
                 PDF
                  ↓
           Document Loader
                  ↓
            Text Splitter
                  ↓
            Embedding Model
                  ↓
             Vector Store
                  ↓
             Retriever
                  ↓
User Question → Retriever
                  ↓
          Relevant Chunks
                  ↓
             Chat Model
                  ↓
             Final Answer
```

Here:

* **Document Loader** → loads the PDF.
* **Text Splitter** → divides the document.
* **Embedding Model** → converts chunks into vectors.
* **Vector Store** → stores vectors.
* **Retriever** → finds relevant chunks.
* **Chat Model** → generates the final answer.

This is the foundation of a typical LangChain-based RAG pipeline.

---

# Important Interview Questions & Answers

## Beginner Questions

### Q1. What is LangChain?

**Answer:**
LangChain is a framework for building applications powered by LLMs. It provides abstractions and integrations for models, prompts, tools, retrieval, agents, and other components.

---

### Q2. What are the major components of LangChain?

**Answer:**
Important components include:

* Models
* Prompts
* Messages
* Tools
* Retrievers
* Document loaders
* Text splitters
* Embeddings
* Vector stores
* Chains
* Agents
* Structured output

---

### Q3. What is a model in LangChain?

**Answer:**
A model is the AI component that processes input and produces an output. LangChain provides standardized interfaces for interacting with models from different providers. ([Docs by LangChain][1])

---

### Q4. What is a chat model?

**Answer:**
A chat model is a model designed to work with conversational messages such as system, human, AI, and tool messages.

---

### Q5. What is an embedding model?

**Answer:**
An embedding model converts text or other supported data into numerical vectors that represent semantic information. These vectors are commonly used for semantic search and RAG.

---

## Intermediate Questions

### Q6. What is the difference between a chat model and an embedding model?

**Answer:**

```text
Chat Model:
Input → Conversation → Output

Embedding Model:
Input → Vector
```

Chat models generate responses, while embedding models create numerical representations for similarity and retrieval.

---

### Q7. What is a tool in LangChain?

**Answer:**
A tool is a callable function that an AI application can use to interact with external systems, APIs, databases, calculators, search engines, or application code.

---

### Q8. What is the difference between a chain and an agent?

**Answer:**

**Chain:**

```text
A → B → C → D
```

The workflow is predetermined.

**Agent:**

```text
          ┌→ Tool A
LLM ──────┼→ Tool B
          └→ Tool C
```

The agent can decide which action/tool to use based on the task.

---

### Q9. Why are embeddings needed in RAG?

**Answer:**
Embeddings convert documents and queries into vectors so that semantically similar information can be retrieved from a vector store.

---

### Q10. What is temperature?

**Answer:**
Temperature controls the randomness/variability of model generation. Lower values generally produce more predictable outputs, while higher values allow more variation.

---

## Scenario-Based Questions

### Q11. You need to build a PDF question-answering application. Which LangChain components would you use?

**Answer:**

```text
PDF
 ↓
Document Loader
 ↓
Text Splitter
 ↓
Embedding Model
 ↓
Vector Store
 ↓
Retriever
 ↓
Chat Model
 ↓
Answer
```

---

### Q12. You need an AI agent that can search the web and query a database. What component would you use?

**Answer:**
Use **tools** for the web search and database operations, and an **agent** to decide when and which tool to call.

---

### Q13. Your application needs semantic search. Which model should you use?

**Answer:**
Use an **embedding model**, because semantic search requires vector representations of queries and documents.

---

### Q14. Your application needs to generate a natural-language answer. Which model should you use?

**Answer:**
Use a **chat model** or other appropriate generative model.

---

### Q15. Your application requires the LLM to return predictable JSON. What should you use?

**Answer:**
Use the model's **structured-output capability** or an appropriate output parser/schema.

---

# 30-Second Revision

```text
LangChain = Framework for building LLM applications.

Major Components:

Models
→ Generate/understand AI responses

Prompts
→ Instructions given to models

Tools
→ Allow AI applications to perform actions

Retrievers
→ Find relevant information

Embeddings
→ Convert text into vectors

Vector Stores
→ Store/search vectors

Chains
→ Fixed sequence of operations

Agents
→ Decide which actions/tools to use

Document Loaders
→ Load external documents

Text Splitters
→ Break documents into chunks

Structured Output
→ Return predictable data formats
```

### Models

```text
Chat Model
→ Input: Messages
→ Output: AI response

Embedding Model
→ Input: Text
→ Output: Vector
```

---

# 2-Minute Revision

## LangChain Components

Think of a LangChain application as:

```text
Data → Retrieval → Model → Tools → Application
```

### Models

The **AI brain** that processes input and generates output.

### Prompts

Instructions that tell the model what to do.

### Tools

Functions/APIs that allow the application to interact with the outside world.

### Retrievers

Find relevant information from a knowledge base.

### Embeddings

Convert information into vectors representing semantic meaning.

### Vector Store

Stores embeddings and performs similarity search.

### Document Loader

Loads PDFs, web pages, files, databases, etc.

### Text Splitter

Breaks large documents into smaller chunks.

### Chains

Predefined workflows:

```text
A → B → C
```

### Agents

Dynamic workflows where the model decides what action/tool to use.

---

## Model Types

### Chat Model

```text
Messages → AI Response
```

Used for:

* Chatbots
* Question answering
* Agents
* Reasoning
* Generation

### Embedding Model

```text
Text → Vector
```

Used for:

* RAG
* Semantic search
* Similarity
* Retrieval

---

## Final Interview Memory

> **LangChain provides the building blocks for LLM applications. Models perform the AI work, prompts provide instructions, tools enable actions, retrievers find information, embeddings convert information into vectors, vector stores enable semantic search, chains define fixed workflows, and agents dynamically choose actions.**

([Docs by LangChain][1])

[1]: https://docs.langchain.com/oss/python/deepagents/models?utm_source=chatgpt.com "Models - Docs by LangChain"
